In [1]:
import pandas as pd
import numpy as np

# ==============================================================================
# Task 1: Merging the CSV Files into a Single Dataframe
# This task primarily involves loading CSVs and merging operations, which are
# best handled by Pandas due to its DataFrame structure and robust merging
# functionalities. NumPy does not offer direct equivalents for these relational
# database-like operations on structured data from files.
# ==============================================================================

print("--- Starting Task 1 ---")

# Task 1.a) Load the contents of the four files into Pandas DataFrame objects
# These DataFrames will be the starting point for all data manipulations.
languages = pd.read_csv('languages.csv')
parameters = pd.read_csv('parameters.csv')
values = pd.read_csv('values.csv')
codes = pd.read_csv('codes.csv')
print("Task 1.a: Loaded CSVs into DataFrames.")

# Task 1.b) Reduce 'values' to parameters GB020, GB022, GB023 and remove columns
# The PDF mentions GB020, GB021, GB022 in Task 1.b, but later tasks (2.c, 1.d example)
# consistently use GB020, GB022, GB023. We will proceed with GB020, GB022, GB023 for consistency.
values = values[values['Parameter_ID'].isin(['GB020', 'GB022', 'GB023'])]
# Remove specified columns from the values DataFrame.
values = values.drop(columns=['ID', 'Comment', 'Source', 'Source_comment', 'Coders'])
print("Task 1.b: Filtered values and removed unnecessary columns.")

# Task 1.c) Merge Name and Macroarea from 'languages' into 'values'
# This merge links language information to the typological values.
values = pd.merge(values, languages[['ID', 'Name', 'Macroarea']],
                  left_on='Language_ID', right_on='ID', how='left')
# Drop the redundant 'ID' column that came from 'languages' after the merge.
values = values.drop(columns=['ID'])
print("Task 1.c: Merged language names and macroareas.")

# Task 1.d) Merge value descriptions from 'codes' into 'values'
# This merge adds human-readable descriptions for the coded values (e.g., 'present', 'absent').
values = pd.merge(values, codes[['ID', 'Description']],
                  left_on='Code_ID', right_on='ID', how='left')
# Drop the redundant 'ID' column that came from 'codes' after the merge.
values = values.drop(columns=['ID'])
print("Task 1.d: Merged value descriptions.")

print("\n--- Task 1 End Result (last 3 rows of values DataFrame) ---")
print(values.tail(3))
print("\n")


# ==============================================================================
# Task 2: Getting the values Dataframe into Shape
# This task involves further cleaning and renaming columns, and sorting. These
# are still DataFrame-level operations. After this task, we will convert the
# DataFrame to NumPy arrays for subsequent NumPy-specific manipulations.
# ==============================================================================

print("--- Starting Task 2 ---")

# Task 2.a) Remove irrelevant columns and rename remaining ones
# We keep 'Parameter_ID' to rename it to 'Feature', and drop original 'Value'
# and 'Code_ID' as 'Description' will become 'Value'.
values = values.drop(columns=['Value', 'Code_ID'])
values = values.rename(columns={
    'Name': 'Language',
    'Description': 'Value',
    'Parameter_ID': 'Feature' # Retained and renamed from original values DataFrame
})
print("Task 2.a: Removed/renamed columns.")

# Task 2.b) Reorder columns and sort the result
# Ensures a consistent structure and order for analysis.
values = values[['Language', 'Macroarea', 'Feature', 'Value']]
values = values.sort_values(by=['Macroarea', 'Language'])
print("Task 2.b: Reordered columns and sorted data.")

# Task 2.c) Remove lines with incomplete data and replace feature names
# `dropna()` handles rows with any missing values.
values = values.dropna()
# `replace()` provides more mnemonic names for the feature IDs.
values['Feature'] = values['Feature'].replace({
    'GB020': 'defArt',
    'GB022': 'prenom',
    'GB023': 'postnom'
})
print("Task 2.c: Removed incomplete data and replaced feature names.")

print("\n--- Task 2 End Result (last 3 rows of values DataFrame) ---")
print(values.tail(3))
print("\n")


# Convert the processed Pandas DataFrame columns into NumPy arrays.
# Explicitly casting to 'U100' (Unicode string of max length 100) ensures
# consistent string handling across all NumPy operations and avoids errors
# with mixed types or varying string lengths.
language_np = values['Language'].to_numpy(dtype='U100')
macroarea_np = values['Macroarea'].to_numpy(dtype='U100')
feature_np = values['Feature'].to_numpy(dtype='U100')
value_np = values['Value'].to_numpy(dtype='U100')
print("Converted DataFrame columns to NumPy arrays for subsequent tasks.")


# ==============================================================================
# Task 3: Hierarchical Indexing (NumPy Implementation)
# Pandas' hierarchical indexing (`MultiIndex`) is a core DataFrame feature.
# In NumPy, we simulate this by using boolean indexing, stacking arrays, and
# careful iteration/lookup, often resulting in more explicit, manual steps.
# ==============================================================================

print("--- Starting Task 3 ---")

# Task 3.a) Modify values by creating a hierarchical row index
# In pure NumPy, we don't have a direct equivalent of Pandas' `set_index` for MultiIndex.
# The data implicitly forms groups based on Macroarea and Language.
# We will work with the flat NumPy arrays and filter them as needed.
print("Task 3.a (NumPy conceptual): Hierarchical indexing is a Pandas MultiIndex feature. "
      "In NumPy, we work with flat arrays and use filtering/grouping logic to simulate it.")


# Task 3.b) Global distribution of definite articles
# This involves filtering by 'feature' and then counting unique 'value' occurrences.
print("\n--- Task 3.b Global Distribution of Definite Articles (NumPy) ---")
# Create a boolean mask to select rows where 'feature' is 'defArt'.
defart_mask_global = (feature_np == 'defArt')
# Apply the mask to 'value_np' to get all values for definite articles.
global_defart_values = value_np[defart_mask_global]
# Use `np.unique` with `return_counts=True` to get unique values and their frequencies,
# similar to Pandas' `value_counts()`.
global_unique, global_counts = np.unique(global_defart_values, return_counts=True)
# Combine unique values and counts into a dictionary for a clear display.
global_defart_distribution_np = dict(zip(global_unique, global_counts))
print("Global Definite Article Distribution:", global_defart_distribution_np)


print("\n--- Task 3.b Eurasia Distribution of Definite Articles (NumPy) ---")
# Filter for 'Eurasia' macroarea and 'defArt' feature.
eurasia_mask = (macroarea_np == 'Eurasia')
defart_mask_eurasia = (feature_np == 'defArt')
# Combine both masks using logical AND to get rows satisfying both conditions.
combined_mask_eurasia = eurasia_mask & defart_mask_eurasia
eurasia_defart_values = value_np[combined_mask_eurasia]
eurasia_unique, eurasia_counts = np.unique(eurasia_defart_values, return_counts=True)
eurasia_defart_distribution_np = dict(zip(eurasia_unique, eurasia_counts))
print("Eurasia Definite Article Distribution:", eurasia_defart_distribution_np)


print("\n--- Task 3.b Africa Distribution of Definite Articles (NumPy) ---")
# Filter for 'Africa' macroarea and 'defArt' feature.
africa_mask = (macroarea_np == 'Africa')
defart_mask_africa = (feature_np == 'defArt')
combined_mask_africa = africa_mask & defart_mask_africa
africa_defart_values = value_np[combined_mask_africa]
africa_unique, africa_counts = np.unique(africa_defart_values, return_counts=True)
africa_defart_distribution_np = dict(zip(africa_unique, africa_counts))
print("Africa Definite Article Distribution:", africa_defart_distribution_np)
print("\n")


# Task 3.c) Reindex the values dataframe with a different hierarchical row index
# This is a conceptual explanation as NumPy does not have MultiIndex reindexing.
print("Task 3.c (NumPy conceptual): Reindexing with different hierarchical levels "
      "is a Pandas MultiIndex operation. In NumPy, you would re-organize or re-sort "
      "the underlying arrays based on the desired 'level' order, or create a new "
      "structured representation.")


# Task 3.d) Extract all languages with prenominal article position
# This involves filtering by 'feature' and 'value' and extracting unique language names.
print("--- Task 3.d Languages with Prenominal Article Position (Value: present) (NumPy) ---")
# Create a mask for 'prenominal' feature.
prenom_mask = (feature_np == 'prenom')
# Create a mask for 'present' value.
present_mask = (value_np == 'present')
# Combine both masks to find rows where 'feature' is 'prenom' AND 'value' is 'present'.
combined_prenom_present_mask = prenom_mask & present_mask
# Apply the combined mask to the 'language_np' array to get the languages that match.
prenominal_present_languages_np = language_np[combined_prenom_present_mask]
# Use `np.unique` to get only the unique language names from the filtered array,
# then convert to a list.
prenominal_present_languages_list_np = np.unique(prenominal_present_languages_np).tolist()
print("Languages with Prenominal Article Position (present):", prenominal_present_languages_list_np)
print("\n")


# ==============================================================================
# Task 4: Pivoting and Melting (NumPy Implementation)
# Pandas `pivot` and `melt` are powerful reshaping tools. In NumPy, these
# operations are manual and require iterating, dictionary building, or careful
# array reshaping and concatenation.
# ==============================================================================

print("--- Starting Task 4 ---")

# Task 4.a & 4.b) Pivot to get features as columns, removing unnecessary levels.
# Simulating the pivot operation by building a dictionary where keys are
# (Macroarea, Language) and values are dictionaries of features and their values.
print("--- Task 4.a & 4.b languages_rows (NumPy conceptual output - head) ---")

# Get unique combinations of Macroarea and Language, which will serve as the
# unique rows in our pivoted output.
unique_macro_lang_combinations = np.unique(np.column_stack((macroarea_np, language_np)), axis=0)

# Initialize a dictionary to store the pivoted data.
# `languages_rows_dict[(macroarea, language)]` will hold `{feature: value}`.
languages_rows_dict = {}

# Define the features that are expected to become columns after pivoting.
features_to_pivot = ['defArt', 'prenom', 'postnom']

# Populate the dictionary by iterating through the original flat NumPy arrays.
for i in range(len(language_np)):
    current_macroarea = macroarea_np[i]
    current_language = language_np[i]
    current_feature = feature_np[i]
    current_value = value_np[i]

    key = (current_macroarea, current_language)
    if key not in languages_rows_dict:
        # If this (macroarea, language) pair is seen for the first time,
        # initialize its feature dictionary with None values.
        languages_rows_dict[key] = {feature: None for feature in features_to_pivot}
    
    # Assign the current feature's value.
    if current_feature in features_to_pivot:
        languages_rows_dict[key][current_feature] = current_value

# To display the pivoted data, we'll extract it into a list of lists,
# which can be conceptually printed like a table.
# Sort the keys (Macroarea, Language) for consistent output order, mimicking Pandas' sorting.
sorted_keys = sorted(languages_rows_dict.keys())

# Prepare header for printing.
header = f"{'Macroarea':<20} {'Language':<20} {'defArt':<10} {'prenom':<10} {'postnom':<10}"
print(header)
print("-" * len(header)) # Separator line

# Print the first few rows of the conceptual pivoted data.
# This simulates the `languages_rows` DataFrame head.
for i in range(min(5, len(sorted_keys))): # Print up to 5 rows or all if fewer than 5
    key = sorted_keys[i]
    # Use .get() with a default of '' for potentially missing values (None).
    row_values = languages_rows_dict[key]
    print(f"{key[0]:<20} {key[1]:<20} {row_values.get('defArt', ''):<10} "
          f"{row_values.get('prenom', ''):<10} {row_values.get('postnom', ''):<10}")
print("Task 4.a & 4.b: Simulated pivot and droplevel operation conceptually.")
print("\n")


# Task 4.c) Languages with definite articles in both postnominal and prenominal position.
# This involves filtering the manually "pivoted" data (languages_rows_dict).
print("--- Task 4.c Languages with Definite Articles in Both Postnominal and Prenominal Position (NumPy) ---")
languages_with_both_prenom_postnom_np = []
for (macroarea, language), features in languages_rows_dict.items():
    # Check if both 'prenom' and 'postnom' features are 'present' for the current language.
    if features.get('prenom') == 'present' and features.get('postnom') == 'present':
        languages_with_both_prenom_postnom_np.append(language)

print("Languages with definite articles in both prenominal and postnominal position:")
print(languages_with_both_prenom_postnom_np)
print("\n")


# Task 4.d) Chained method calls to transform back to original 'values' format (conceptual).
# This is the inverse of pivoting, effectively a "melt" operation.
# In NumPy, this requires manually reconstructing the long format from the "pivoted" data.
print("--- Task 4.d Chained method calls to revert (Conceptual NumPy steps) ---")
print("To transform the conceptual 'languages_rows' back to the 'values' format (long format) in NumPy, "
      "you would perform the following conceptual steps:")
print("1. Iterate through each (Macroarea, Language) entry in the pivoted structure (e.g., `languages_rows_dict`).")
print("2. For each (Macroarea, Language) pair, iterate through its associated feature-value pairs (e.g., {'defArt': 'present'}).")
print("3. For each feature-value pair, construct a new record (Macroarea, Language, Feature_Name, Feature_Value).")
print("4. Collect all such records into a new flat list or array.")
print("5. Sort this new structure by Macroarea, then Language, to match the original 'values' sorting.")
print("This effectively simulates Pandas' `.reset_index().melt(...).sort_values()` chain manually.")
print("\n")


# ==============================================================================
# Task 5: Concatenation (NumPy Implementation)
# NumPy's `np.concatenate` works directly on arrays. Simulating Pandas'
# `pd.concat` with `keys` (for hierarchical index) or `join='inner'` (for
# column-wise merge) requires more manual data preparation and alignment.
# ==============================================================================

print("--- Starting Task 5 ---")

# Task 5.a) Extract three dataframes (conceptual arrays) for each feature.
# This is achieved by simple boolean indexing on the `feature_np` array.
print("--- Task 5.a Feature-specific Data (NumPy representation - head) ---")

def_art_mask = (feature_np == 'defArt')
prenom_mask = (feature_np == 'prenom')
postnom_mask = (feature_np == 'postnom')

# `np.column_stack` combines 1D arrays into a 2D array, representing the conceptual columns.
# Note: The order of columns here is (Language, Macroarea, Feature, Value).
def_art_data_np = np.column_stack((language_np[def_art_mask], macroarea_np[def_art_mask], feature_np[def_art_mask], value_np[def_art_mask]))
prenom_data_np = np.column_stack((language_np[prenom_mask], macroarea_np[prenom_mask], feature_np[prenom_mask], value_np[prenom_mask]))
postnom_data_np = np.column_stack((language_np[postnom_mask], macroarea_np[postnom_mask], feature_np[postnom_mask], value_np[postnom_mask]))

print("def_art_data head (Language, Macroarea, Feature, Value):")
print(def_art_data_np[:min(5, len(def_art_data_np))])
print("\nprenom_data head:")
print(prenom_data_np[:min(5, len(prenom_data_np))])
print("\npostnom_data head:")
print(postnom_data_np[:min(5, len(postnom_data_np))])
print("Task 5.a: Extracted feature-specific data using boolean masks.")
print("\n")


# Task 5.b) Concatenate along the column axis, limiting to complete rows (inner join).
# In NumPy, an "inner join" is implemented by finding the intersection of unique keys
# (Macroarea, Language) across the datasets and then combining the corresponding values.
print("--- Task 5.b Concatenated along column axis (inner join, NumPy conceptual output - head) ---")

# Convert the feature-specific data into dictionaries for easier lookup by (Macroarea, Language) key.
# The `r[1]` is Macroarea, `r[0]` is Language, `r[3]` is Value from the stacked array.
def_art_map = {(row[1], row[0]): row[3] for row in def_art_data_np}
prenom_map = {(row[1], row[0]): row[3] for row in prenom_data_np}
postnom_map = {(row[1], row[0]): row[3] for row in postnom_data_np}

# Find the intersection of keys (Macroarea, Language tuples) present in all three maps.
# This simulates the `join='inner'` behavior.
common_keys = set(def_art_map.keys()) \
              .intersection(set(prenom_map.keys())) \
              .intersection(set(postnom_map.keys()))

# Sort the common keys to ensure consistent output order.
sorted_common_keys = sorted(list(common_keys))

# Construct the list of lists for the concatenated output.
concatenated_columns_complete_np_list = []
for macroarea, language in sorted_common_keys:
    concatenated_columns_complete_np_list.append([
        macroarea,
        language,
        def_art_map[(macroarea, language)], # Value for 'defArt'
        prenom_map[(macroarea, language)],  # Value for 'prenom'
        postnom_map[(macroarea, language)]  # Value for 'postnom'
    ])

# Print the conceptual concatenated output, structured as a table.
header = f"{'Macroarea':<20} {'Language':<20} {'defArt':<10} {'prenom':<10} {'postnom':<10}"
print(header)
print("-" * len(header))
for row in concatenated_columns_complete_np_list[:min(5, len(concatenated_columns_complete_np_list))]:
    print(f"{row[0]:<20} {row[1]:<20} {row[2]:<10} {row[3]:<10} {row[4]:<10}")
print("Task 5.b: Simulated column-wise concatenation with inner join logic.")
print("\n")


# Task 5.c) Concatenate along the row axis with an additional outermost hierarchical index.
# In NumPy, this means creating a new combined structure where a 'feature key' becomes an
# implicit outer level, followed by Macroarea and Language. We achieve this by creating
# tuples and then sorting them to represent the hierarchy.
print("--- Task 5.c Series with Three-Level Hierarchical Index (NumPy conceptual output - head) ---")

# Create a list of tuples where each tuple represents a row in the desired hierarchical structure:
# (Feature_Key, Macroarea, Language, Value).
series_with_hierarchical_index_np_list = []

# Iterate through the original flat NumPy arrays to reconstruct the desired records.
for i in range(len(language_np)):
    current_language = language_np[i]
    current_macroarea = macroarea_np[i]
    current_feature = feature_np[i]
    current_value = value_np[i]
    series_with_hierarchical_index_np_list.append(
        (current_feature, current_macroarea, current_language, current_value)
    )

# Sort the list of tuples. Python's default tuple sorting will naturally create the
# hierarchical order: sort first by Feature, then by Macroarea, then by Language.
series_with_hierarchical_index_np_list.sort(key=lambda x: (x[0], x[1], x[2]))

# Print the conceptual series with a three-level hierarchical index.
header = f"{'Feature':<10} {'Macroarea':<20} {'Language':<20} {'Value':<10}"
print(header)
print("-" * len(header))
for row in series_with_hierarchical_index_np_list[:min(5, len(series_with_hierarchical_index_np_list))]:
    print(f"{row[0]:<10} {row[1]:<20} {row[2]:<20} {row[3]:<10}")
print("Task 5.c: Simulated row-wise concatenation with conceptual hierarchical indexing.")
print("\n")


--- Starting Task 1 ---
Task 1.a: Loaded CSVs into DataFrames.
Task 1.b: Filtered values and removed unnecessary columns.
Task 1.c: Merged language names and macroareas.
Task 1.d: Merged value descriptions.

--- Task 1 End Result (last 3 rows of values DataFrame) ---
     Language_ID Parameter_ID Value  Code_ID  Name      Macroarea Description
7146    zuni1245        GB020     1  GB020-1  Zuni  North America     present
7147    zuni1245        GB022     0  GB022-0  Zuni  North America      absent
7148    zuni1245        GB023     0  GB023-0  Zuni  North America      absent


--- Starting Task 2 ---
Task 2.a: Removed/renamed columns.
Task 2.b: Reordered columns and sorted data.
Task 2.c: Removed incomplete data and replaced feature names.

--- Task 2 End Result (last 3 rows of values DataFrame) ---
     Language      Macroarea  Feature    Value
6914   Yámana  South America   defArt   absent
6915   Yámana  South America   prenom   absent
6916   Yámana  South America  postnom  present


C